# Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
sns.set(style="whitegrid", context="notebook")

from sklearn.model_selection import train_test_split
from datetime import datetime
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings("ignore")

print(tf.version.VERSION)
print(keras.__version__)

!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/models.py" -P models -nc
!wget "https://raw.githubusercontent.com/ibrahimerdem/application_storage/main/aux_func.py" -P aux_func -nc
import sys
sys.path.append("models")
sys.path.append("aux_func")

from aux_func import *
from models import *

In [ ]:
# !!! Includes link for download  

!wget "https://github.com/ibrahimerdem/application_storage/raw/main/ecommerce_dataset.zip"
!unzip "ecommerce_dataset.zip"
data = pd.read_csv("ecommerce_dataset.csv")

# Data Preparing

In [ ]:
df = data.copy()

In [ ]:
df = df[df["status"]=="complete"]
df = df.iloc[:, [0, 2, 20, 3, 8, 4, 5, 11]]
df.rename(columns={"item_id":"t_id",
                   "created_at":"t_date",
                   "Customer ID":"c_id",
                   "sku":"item_name",
                   "category_name_1":"item_category",
                   "qty_ordered":"amount"}, inplace=True)
df["t_date"] = df["t_date"].astype("datetime64")

c_freq = 2
p_freq = 2
m_freq = 1000
t_freq = 2500

freq = pd.DataFrame(df["item_name"].value_counts())
f_items = freq[freq["item_name"]>p_freq].index
df = df[df["item_name"].isin(f_items)]
print("after first filter")
print("#customers:", df["c_id"].nunique(),
      "#items", df["item_name"].nunique())

c_data = df.groupby("c_id", as_index=False).agg({"t_id":"count"})
f_cust = c_data.loc[c_data["t_id"]>c_freq, "c_id"].values
df = df[df["c_id"].isin(f_cust)]

fmet = pd.DataFrame(df["payment_method"].value_counts())
f_met = fmet.loc[fmet["payment_method"]<m_freq, ["payment_method"]]
df.loc[df["payment_method"].isin(f_met.index), "payment_method"] = "other"

fcat = pd.DataFrame(df["item_category"].value_counts())
f_cat = fcat.loc[fcat["item_category"]<t_freq, ["item_category"]]
df.loc[df["item_category"].isin(f_cat.index), "item_category"] = "other"

print("\nafter second filter")
mind = df["t_date"].min()
maxd = df["t_date"].max()
print("#customers:", df["c_id"].nunique(),
      "#items", df["item_name"].nunique())

print("\ndate between:", mind.day, mind.month, mind.year,
      "-", maxd.day, maxd.month, maxd.year)
print("#transactions:", df["t_id"].nunique())
print("#product-categories:", df["item_category"].nunique())
print("#payment-methods:", df["payment_method"].nunique())

df["recency"] = df["t_date"].max() - df["t_date"]
df["recency"] = df["recency"].dt.days

df.loc[df["price"]==0, "price"] = 0.1
df["payment"] = df["price"]*df["amount"]
df.drop(columns=["price", "amount"], inplace=True)

df["recency"] = ((df["recency"]*99)/df["recency"].max())+1
df["recency"] = df["recency"].astype("int64")

df["payment"] = ((df["payment"]*999)/df["payment"].max())+1
df["payment"] = df["payment"].astype("int64")

df["month"] = df["t_date"].dt.month
df["month"] = df["month"].astype("int64")

df["dayofweek"] = df["t_date"].dt.dayofweek
df["dayofweek"] = df["dayofweek"].astype("int64") + 1

cust = df["c_id"].unique()
c_size = len(cust)
cust2code = {}
code2cust = {}
for num, c in enumerate(cust):
    cust2code[c] = num+1
    code2cust[num+1] = c

df["c_id"] = df["c_id"].map(cust2code)

items = df["item_name"].unique()
i_size = len(items)
item2code = {}
code2item = {}
for num, item in enumerate(items):
    item2code[item] = num+1
    code2item[num+1] = item

df["item_code"] = df["item_name"].map(item2code)

cats = df["item_category"].unique()
c_size = len(cats)
cat2code = {}
code2cat = {}
for num, cat in enumerate(cats):
    cat2code[cat] = num+1
    code2cat[num+1] = cat

df["category_code"] = df["item_category"].map(cat2code)

methods = df["payment_method"].unique()
m_size = len(methods)
method2code = {}
code2method = {}
for num, method in enumerate(methods):
    method2code[method] = num+1
    code2method[num+1] = method

df["method_code"] = df["payment_method"].map(method2code)
df = df.drop(columns=["item_name", "item_category", "payment_method"])

In [ ]:
df.to_csv("data_set.csv",
          sep=";",
          index=False)

# Modeling

In [ ]:
processed_data = pd.read_csv("data_set.csv",
                             sep=";")

## Sequence Creation

In [ ]:
df_a = processed_data.copy()

model_start = "07-01-2016"
model_end = "04-01-2018"
test_start = "04-01-2018"
test_end = "07-01-2018"

df_a["t_date"] = df_a["t_date"].astype("datetime64")
df_base = df_a[(df_a["t_date"]>=model_start)&(df_a["t_date"]<model_end)]

pro_max = df_base["item_code"].astype("int64").max()
rec_max = df_base["recency"].astype("int64").max()
pay_max = df_base["payment"].astype("int64").max()
cat_max = df_base["category_code"].astype("int64").max()
met_max = df_base["method_code"].astype("int64").max()
mon_max = df_base["month"].astype("int64").max()
day_max = df_base["dayofweek"].astype("int64").max()

n_padded = 32
batch_size = 128
val_split = 0.2

hold_df = df_a.loc[(df_a["t_date"]>=test_start)&(df_a["t_date"]<test_end),
                   ["c_id", "item_code"]]
items = df_base["item_code"].unique()
hold_df = hold_df[hold_df["item_code"].isin(items)]
hold_c = hold_df["c_id"].unique()

train = df_base[~(df_base["c_id"].isin(hold_c))]
test = df_base[df_base["c_id"].isin(hold_c)]

print("train customers:", train["c_id"].nunique())
print("test customers:", test["c_id"].nunique())

train_data = create_sequences(train)
train_data = create_splits(train_data)
train_list = make_padding(train_data, n_padded)

test_data = create_sequences(test)
test_data.rename(columns={"item_code":"input_seq"}, inplace=True)
test_list = make_padding(test_data, n_padded)

## Training

In [ ]:
#change model number 1-8

hparams = {
    "bidirect": False,
    "style": "lstm",
    "model": 7,
    "emb_unit": 64,
    "rnn_unit": 200,
    "dropout": 0.8,
    "learning": 0.001,
    "n_padded": n_padded,
    "item_max": pro_max+1,
    "rec_max": rec_max+1,
    "pay_max": pay_max+1,
    "cat_max": cat_max+1,
    "met_max": met_max+1,
    "mon_max": mon_max+1,
    "day_max": day_max+1,
}

In [ ]:
best_model = model_base(hparams)

hist = model_training(best_model, [train_list[0], train_list[1],
                                    train_list[5], train_list[6],
                                    train_list[3]],
                                   train_list[-1],
                                   val_split, verbose=1)

best_model.save("trained_model.h5")

# Marketable Products

In [ ]:
# !!! Includes link for download  

!wget "https://github.com/ibrahimerdem/application_storage/raw/main/trained_model.zip"
!unzip "trained_model.zip"

best_model = keras.models.load_model(
    "trained_model.h5",
    custom_objects={"loss_function":loss_function})

y_pred = best_model.predict([test_list[0], test_list[1],
                             test_list[5], test_list[6],
                             test_list[3]])

customer_list = test_data["c_id"].unique()

main_table = hold_df[hold_df["c_id"].isin(customer_list)]

main_table["hit"] = 0
main_table["r_list"] = 0
counter = len(customer_list)
for i in range(counter):
    c = customer_list[i]
    rel = set(main_table.loc[main_table["c_id"]==c, "item_code"].values)
    if len(rel) > 0:
        ret = y_pred[i][-1].argsort()[::-1][:5]
        rlist = ",".join([str(i) for i in ret])
        main_table.loc[main_table["c_id"]==c, "r_list"] = rlist 
        inter = set.intersection(rel, ret)
        if len(inter) > 0:
            main_table.loc[main_table["c_id"]==c, "hit"] = 1          

result = main_table.groupby("c_id").agg(
    {"item_code": lambda x: ",".join([str(i) for i in x]),
     "hit": "max",
     "r_list": "last"})

In [ ]:
result

In [ ]:
result[result["hit"]==1]